## Logistic regression 
   1. predict `is_canceled`


### 1.Load data

In [8]:
import sys
sys.path.append("../src")
from features import clean, add_features, CATEGORICAL, NUMERIC, FEATURES

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/hotels.csv")
df_clean = clean(df)
dff = add_features(df_clean)
print(df.shape)

(119390, 32)


In [9]:
dff

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,nights,arrival_month,arrival_weekday,is_agent,is_company
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,0,0,Check-Out,2015-07-01,2015-07-01,0,7,2,0,0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,0,0,Check-Out,2015-07-01,2015-07-01,0,7,2,0,0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,0,Check-Out,2015-07-02,2015-07-01,1,7,2,0,0
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,0,Check-Out,2015-07-02,2015-07-01,1,7,2,1,0
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,1,Check-Out,2015-07-03,2015-07-01,2,7,2,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87391,City Hotel,0,23,2017,August,35,30,2,5,2,...,0,0,Check-Out,2017-09-06,2017-08-30,7,8,2,1,0
87392,City Hotel,0,102,2017,August,35,31,2,5,3,...,0,2,Check-Out,2017-09-07,2017-08-31,7,8,3,1,0
87393,City Hotel,0,34,2017,August,35,31,2,5,2,...,0,4,Check-Out,2017-09-07,2017-08-31,7,8,3,1,0
87394,City Hotel,0,109,2017,August,35,31,2,5,2,...,0,0,Check-Out,2017-09-07,2017-08-31,7,8,3,1,0


### 2. Split the Data using the split_period from contract.py 

In [ ]:
import sys
sys.path.append("../src")    
from notes.contract import split_periods  

train, val, test = split_periods(dff)
print(f'Train Data:{len(train)} \nValidation Data:{len(val)} \nTest Data:{len(test)}')            
#print(train["is_canceled"].mean())                

Train Data:32866 
Validation Data:22838 
Test Data:31692


In [5]:
x_train, y_train = train[FEATURES], train['is_canceled']
x_val,   y_val   = val[FEATURES],   val['is_canceled']
x_test,  y_test  = test[FEATURES],  test['is_canceled']

In [7]:
x_train.info()

<class 'pandas.DataFrame'>
Index: 32866 entries, 0 to 74336
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype
---  ------                          --------------  -----
 0   hotel                           32866 non-null  str  
 1   market_segment                  32866 non-null  str  
 2   distribution_channel            32866 non-null  str  
 3   deposit_type                    32866 non-null  str  
 4   customer_type                   32866 non-null  str  
 5   country                         32866 non-null  str  
 6   reserved_room_type              32866 non-null  str  
 7   arrival_month                   32866 non-null  str  
 8   arrival_weekday                 32866 non-null  str  
 9   lead_time                       32866 non-null  int64
 10  is_repeated_guest               32866 non-null  int64
 11  previous_cancellations          32866 non-null  int64
 12  previous_bookings_not_canceled  32866 non-null  int64
 13  is_agent         

### Encode and train the model
- categorical columns -> one-hot (unseen categories in later periods are ignored)
- numeric columns -> standardised so regularisation treats them fairly

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from features import clean, add_features, CATEGORICAL, NUMERIC, FEATURES

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ("num", StandardScaler(), NUMERIC),
])

model = Pipeline([("pre", pre), ("lr", LogisticRegression(max_iter=1000))])
model.fit(x_train, y_train)      # learn encoding + coefficients on train only

In [ ]:
# Evaluate on train and validation (test stays untouched until the final step)
from sklearn.metrics import accuracy_score, classification_report
y_pred = model.predict(X_val)
print("Train accuracy:", model.score(X_train, y_train))
print("Val accuracy:  ", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

## 5. Check it on validation
- **AUC**: how well it ranks cancellers above non-cancellers (0.5 = coin flip, 1 = perfect).
- **Brier score**: average squared error of the probabilities (lower is better). Compare with the always-predict-the-base-rate baseline.

In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve

p_val = model.predict_proba(X_val)[:, 1]       # column 1 = probability of cancelling
print("AUC  :", roc_auc_score(y_val, p_val))
print("Brier:", brier_score_loss(y_val, p_val))
print("Brier of baseline (always base rate):",
      brier_score_loss(y_val, np.full(len(y_val), y_train.mean())))

## 6. Tune `C` on validation
Try several regularisation strengths and keep the best by validation AUC. Never tune on test.

In [ ]:
results = {}
for C in [0.01, 0.1, 1, 10]:
    m = Pipeline([("pre", pre), ("lr", LogisticRegression(C=C, max_iter=1000))])
    m.fit(X_train, y_train)                                      # refit with this C
    results[C] = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])  # score on validation

print(results)
best_C = max(results, key=results.get)                           # C with the highest AUC
print("best C:", best_C)

## 7. Is the probability trustworthy? (calibration)
For the overbooking number you will **add up** probabilities per night, so they must be honest. If the curve sits on the diagonal, "30% cancel" really means about 30 in 100 cancel.

In [ ]:
frac_pos, mean_pred = calibration_curve(y_val, p_val, n_bins=10)   # actual rate vs predicted, in 10 bins

plt.plot(mean_pred, frac_pos, marker="o", label="model")
plt.plot([0, 1], [0, 1], "--", label="perfect")                    # the ideal diagonal
plt.xlabel("predicted probability"); plt.ylabel("actual cancel rate")
plt.legend(); plt.show()

## 8. Read the coefficients (what drives cancellation?)

In [ ]:
names = model.named_steps["pre"].get_feature_names_out()         # column names after encoding
coefs = model.named_steps["lr"].coef_[0]                         # one weight per column

imp = pd.Series(coefs, index=names).sort_values()
print(imp.head(10))     # most negative: lowers cancellation odds
print(imp.tail(10))     # most positive: raises cancellation odds (look for deposit_type_Non Refund)

## 9. Final: refit with best C and score on test (do this once)

In [ ]:
final = Pipeline([("pre", pre), ("lr", LogisticRegression(C=best_C, max_iter=1000))])
final.fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))   # use train + validation now that settings are chosen

p_test = final.predict_proba(X_test)[:, 1]
print("TEST AUC:", roc_auc_score(y_test, p_test))

## Next step
Per hotel-night, add `p` over the bookings to get expected cancellations, then set the oversell number `k` from that with a safety buffer chosen on validation using `src/contract.py`.